# Lab 6: EEG raw signal analysis
Records: eyes open/close (5 cycles of 5 s) and fixed-point gaze (30 s).
The `EEG-ABRE-CIERRA-ABIERTOS` file is intentionally excluded.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import trapezoid

DATA_DIR = Path("Data")
FILES = {
    "Eyes open/close (5x5 s)": "EEG-ABRE-CIERRA-09-25_12-01-04.txt",
    "Fixed point (30 s)": "EEG-FIJO-09-25_12-07-18.txt",
}

# BITalino EEG transfer function constants
VCC = 3.3        # supply voltage (V)
G_EEG = 41782    # sensor gain

# Classic EEG bands (Hz)
BANDS = {"Delta": (0.5, 4), "Theta": (4, 8), "Alpha": (8, 13), "Beta": (13, 30), "Gamma": (30, 45)}

## 1. Read OpenSignals file and convert ADC to µV

In [ ]:
def read_opensignals(path):
    """Read an OpenSignals .txt file and return the EEG channel in µV plus metadata."""
    with open(path, encoding="utf-8") as f:
        f.readline()                                   # line 1: file format info
        header = json.loads(f.readline().lstrip("#").strip())  # line 2: JSON metadata
    meta = next(iter(header.values()))                 # metadata of the (only) device

    data = np.loadtxt(path, comments="#")              # skips all header lines
    columns = meta["column"]
    fs = meta["sampling rate"]

    # Locate the EEG channel through the sensor list; fall back to the first analog channel
    sensors, labels = meta["sensor"], meta["label"]
    idx = next((i for i, s in enumerate(sensors) if "EEG" in s.upper()), 0)
    col = columns.index(labels[idx])

    res = meta["resolution"]
    n_bits = res[col] if len(res) == len(columns) else res[idx]

    adc = data[:, col]
    eeg_uv = ((adc / 2**n_bits) - 0.5) * VCC / G_EEG * 1e6   # BITalino EEG transfer function
    t = np.arange(len(adc)) / fs
    return {"t": t, "eeg": eeg_uv, "adc": adc, "fs": fs, "n_bits": n_bits, "label": labels[idx]}

records = {name: read_opensignals(DATA_DIR / fname) for name, fname in FILES.items()}
for name, r in records.items():
    print(f"{name}: fs = {r['fs']} Hz | channel {r['label']} | {r['n_bits']} bits | "
          f"duration = {r['t'][-1]:.1f} s | samples = {len(r['eeg'])}")

## 2. Descriptive statistics and saturation check

In [ ]:
for name, r in records.items():
    x, adc_max = r["eeg"], 2**r["n_bits"] - 1
    clipped = np.mean((r["adc"] <= 0) | (r["adc"] >= adc_max)) * 100
    print(f"{name}\n  mean = {x.mean():.2f} µV | std = {x.std():.2f} µV | "
          f"min = {x.min():.2f} µV | max = {x.max():.2f} µV | saturated samples = {clipped:.2f} %")

## 3. Raw signal in the time domain

In [ ]:
SHOW_5S_GRID = True   # dashed lines every 5 s as a rough reference for the open/close cycles

fig, axes = plt.subplots(len(records), 1, figsize=(14, 3.5 * len(records)))
for ax, (name, r) in zip(np.atleast_1d(axes), records.items()):
    ax.plot(r["t"], r["eeg"], lw=0.5)
    if SHOW_5S_GRID and "open/close" in name:
        for tc in np.arange(0, r["t"][-1], 5):
            ax.axvline(tc, color="gray", ls="--", lw=0.6)
    ax.set(title=f"Raw EEG: {name}", xlabel="Time (s)", ylabel="Amplitude (µV)")
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Power spectral density (Welch)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
psd = {}
for name, r in records.items():
    f, pxx = signal.welch(r["eeg"], fs=r["fs"], nperseg=int(2 * r["fs"]))  # 2 s windows -> 0.5 Hz resolution
    psd[name] = (f, pxx)
    ax.semilogy(f, pxx, label=name)

colors = ["#ddd", "#cde", "#fdd", "#dfd", "#eee"]
for (band, (lo, hi)), c in zip(BANDS.items(), colors):
    ax.axvspan(lo, hi, color=c, alpha=0.5, label=band)
ax.set(xlim=(0, 60), xlabel="Frequency (Hz)", ylabel="PSD (µV²/Hz)", title="Welch PSD of the full recording")
ax.legend(ncol=2); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

## 5. Spectrogram (time-frequency)

In [ ]:
fig, axes = plt.subplots(len(records), 1, figsize=(14, 3.5 * len(records)))
for ax, (name, r) in zip(np.atleast_1d(axes), records.items()):
    f, t, sxx = signal.spectrogram(r["eeg"], fs=r["fs"], nperseg=int(r["fs"]), noverlap=int(0.75 * r["fs"]))
    keep = f <= 40
    im = ax.pcolormesh(t, f[keep], 10 * np.log10(sxx[keep] + 1e-12), shading="gouraud", cmap="viridis")
    ax.axhspan(8, 13, color="white", alpha=0.15)   # alpha band highlight
    ax.set(title=f"Spectrogram: {name}", xlabel="Time (s)", ylabel="Frequency (Hz)")
    fig.colorbar(im, ax=ax, label="Power (dB)")
plt.tight_layout(); plt.show()

## 6. Relative band power

In [ ]:
def band_powers(f, pxx):
    total = trapezoid(pxx[(f >= 0.5) & (f <= 45)], f[(f >= 0.5) & (f <= 45)])
    return {b: trapezoid(pxx[(f >= lo) & (f < hi)], f[(f >= lo) & (f < hi)]) / total * 100
            for b, (lo, hi) in BANDS.items()}

print(f"{'Record':28s}" + "".join(f"{b:>9s}" for b in BANDS))
for name, (f, pxx) in psd.items():
    bp = band_powers(f, pxx)
    print(f"{name:28s}" + "".join(f"{v:8.1f}%" for v in bp.values()))